# Notebook 5: KRONOS Foundation Model Analysis

## Overview

This notebook integrates [KRONOS](https://github.com/mahmoodlab/KRONOS), a panel-agnostic foundation model for spatial proteomics, with KINTSUGI's processing pipeline. KRONOS was trained via self-supervised learning (DINO) on **47 million single-marker patches** spanning 175 protein markers, 16 tissue types, 8 imaging platforms, and 5 institutions.

**What KRONOS provides:**
- Rich, pre-trained embeddings that capture morphological and protein expression patterns
- Label-efficient cell phenotyping (5-10 labels per class)
- Unsupervised tissue microenvironment discovery
- Cross-dataset spatial search and patient stratification

**Prerequisites:**
1. Complete the KINTSUGI processing pipeline through registration (Notebooks 1-2)
2. Install KRONOS: `git clone https://github.com/mahmoodlab/KRONOS.git && pip install -e KRONOS`
3. Request model access on [HuggingFace](https://huggingface.co/MahmoodLab/KRONOS) (CC-BY-NC-ND-4.0)

**Pipeline position:**
```
Raw → Stitch → Decon → EDF → Registration → [KRONOS Analysis] → Phenotyping & Stratification
```

---
## 0. Setup and Configuration

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# ── Project Configuration ─────────────────────────────────────────
PROJECT_DIR = Path(".").resolve()  # Change to your project directory
KRONOS_CACHE = PROJECT_DIR / "model_assets"  # Where KRONOS weights are cached
OUTPUT_DIR = PROJECT_DIR / "data" / "processed" / "embeddings"

# Ensure KINTSUGI is importable
KINTSUGI_DIR = Path(os.environ.get("KINTSUGI_DIR", str(PROJECT_DIR.parent)))
if str(KINTSUGI_DIR / "notebooks") not in sys.path:
    sys.path.insert(0, str(KINTSUGI_DIR / "notebooks"))

print(f"Project: {PROJECT_DIR}")
print(f"KRONOS cache: {KRONOS_CACHE}")
print(f"Output: {OUTPUT_DIR}")

In [ ]:
# ── Dependency Check ──────────────────────────────────────────────
from kintsugi.kronos import _check_kronos_available, _check_torch_available

print(f"PyTorch available: {_check_torch_available()}")
print(f"KRONOS available: {_check_kronos_available()}")

if not _check_kronos_available():
    print("\n⚠ KRONOS not installed. Install with:")
    print("  git clone https://github.com/mahmoodlab/KRONOS.git")
    print("  cd KRONOS && pip install -e .")
    print("\nModel weights require HuggingFace access:")
    print("  https://huggingface.co/MahmoodLab/KRONOS")

if _check_torch_available():
    import torch
    print(f"\nCUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"GPU memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

---
## 1. Marker Mapping

Map KINTSUGI channel names (from `CHANNELNAMES.txt`) to KRONOS's vocabulary of 175 protein markers. Matched markers use KRONOS's pre-computed normalization statistics; unmatched markers use dataset-level normalization.

In [ ]:
from kintsugi.kronos.markers import MarkerMapper

# Create mapper from project
mapper = MarkerMapper.from_project(
    PROJECT_DIR,
    cache_dir=KRONOS_CACHE,
    skip_blank=True,   # Skip 'Blank' and 'Empty' channels
    skip_dapi=False,    # Include DAPI (set True to exclude nuclear stain)
)

print(mapper.summary())

In [ ]:
# ── Review mappings ───────────────────────────────────────────────
# Inspect individual mappings if needed
for m in mapper.mappings:
    status = "✓" if m.matched else "✗"
    print(f"  {status} Cyc{m.cycle} CH{m.channel_idx}: {m.kintsugi_name}"
          f" → {m.kronos_name or 'UNMATCHED'}"
          f" (mean={m.mean:.3f}, std={m.std:.3f})")

---
## 2. Load KRONOS Model

In [ ]:
from kintsugi.kronos.model import KronosModel, KronosConfig

config = KronosConfig(
    model_type="vits16",       # ViT-Small/16 (default, 175 MB, embed_dim=384)
    patch_size=224,            # KRONOS default (ViT-S/16 trained on 224x224)
    token_overlap=False,       # Set True for denser token coverage (stride 8 vs 16)
    cache_dir=str(KRONOS_CACHE),
    device="auto",             # 'auto', 'cuda', or 'cpu'
    batch_size=32,             # Adjust based on GPU memory
    # hf_auth_token="hf_...", # Uncomment if needed for gated access
)

model = KronosModel(config)
model.load()

print(f"Model loaded: precision={model.precision}, embedding_dim={model.embedding_dim}")
print(f"Device: {model.device}")

---
## 3. Extract Embeddings

Load registered images, tile into 224x224 patches, apply two-step normalization (intensity → z-score), pass marker IDs for sinusoidal encoding, and run KRONOS inference. This produces three embedding types:

- **Patch embeddings** `(N, 384)`: CLS token — aggregated representation per patch. Best for clustering and search.
- **Marker embeddings** `(N, M, 384)`: Per-marker features, spatially averaged. Best for marker-specific analysis.
- **Token embeddings** `(N, M, 14, 14, 384)`: Full spatial-molecular resolution. Best for fine-grained spatial analysis.

In [ ]:
from kintsugi.kronos.embeddings import KronosEmbedder

embedder = KronosEmbedder(config=config, model=model)

# Extract embeddings from the entire project
# Two-step normalization is applied automatically:
#   1. Intensity: raw 16-bit / 65535 → [0, 1]
#   2. Z-score: (x - mean) / std per marker (from marker_metadata.csv)
# Marker IDs from KRONOS vocabulary are passed for sinusoidal encoding.
output_path = OUTPUT_DIR / "kronos_embeddings.h5"
result = embedder.embed_project(
    PROJECT_DIR,
    mapper,
    patch_size=224,        # KRONOS default (ViT-S/16)
    overlap=56,            # Pixels of overlap between adjacent patches
    output_path=output_path,
)

print(f"\nExtracted embeddings:")
print(f"  Patches: {result.n_patches}")
print(f"  Embedding dim: {result.embedding_dim}")
print(f"  Patch embeddings: {result.patch_embeddings.shape}")
print(f"  Marker embeddings: {result.marker_embeddings.shape}")
print(f"  Token embeddings: {result.token_embeddings.shape}")
print(f"  Saved to: {output_path}")

In [ ]:
# ── Or load previously saved embeddings ─────────────────────────
# from kintsugi.kronos.model import EmbeddingResult
# result = EmbeddingResult.load(OUTPUT_DIR / "kronos_embeddings.h5")
# print(f"Loaded {result.n_patches} patch embeddings")

---
## 4. Clustering and Visualization

Cluster the patch embeddings to discover tissue microenvironments and cell neighborhoods.

In [ ]:
from kintsugi.kronos.analysis import (
    cluster_embeddings,
    reduce_dimensions,
    visualize_embeddings,
)

# ── Leiden Clustering ─────────────────────────────────────────────
labels = cluster_embeddings(
    result,
    method="leiden",
    resolution=1.0,      # Higher = more clusters
    n_neighbors=15,
)

n_clusters = len(np.unique(labels))
print(f"Found {n_clusters} clusters")
for i in range(n_clusters):
    count = np.sum(labels == i)
    print(f"  Cluster {i}: {count} patches ({100*count/len(labels):.1f}%)")

In [ ]:
# ── UMAP Visualization with Spatial Map ──────────────────────────
fig = visualize_embeddings(
    result,
    labels=labels,
    method="umap",
    show_spatial=True,
    figsize=(16, 6),
    save_path=OUTPUT_DIR / "kronos_clusters.png",
)
plt.show()

In [ ]:
# ── Alternative: KMeans with specified number of clusters ────────
labels_km = cluster_embeddings(
    result,
    method="kmeans",
    n_clusters=8,
)

fig = visualize_embeddings(
    result,
    labels=labels_km,
    method="umap",
    show_spatial=True,
    figsize=(16, 6),
)
plt.suptitle("KMeans (k=8)", y=1.02)
plt.show()

---
## 5. Spatial Search

Find patches similar to a query region across the dataset. Select a region of interest and retrieve the most similar patches by embedding distance.

In [ ]:
from kintsugi.kronos.analysis import spatial_search

# ── Select a query patch ──────────────────────────────────────────
# Pick a patch index (e.g., one from an interesting cluster)
query_idx = 0  # Change to a patch of interest
query_embedding = result.patch_embeddings[query_idx]
query_coords = result.patch_coords[query_idx]

print(f"Query patch {query_idx} at coordinates: {query_coords}")
print(f"Query cluster: {labels[query_idx]}")

# ── Find similar patches ──────────────────────────────────────────
matches = spatial_search(
    query_embedding,
    result,
    top_k=10,
    metric="cosine",
)

print(f"\nTop 10 similar patches:")
for i, m in enumerate(matches):
    print(f"  {i+1}. Patch {m['index']} at {m['coords']} "
          f"(distance={m['distance']:.4f}, cluster={labels[m['index']]})")

In [ ]:
# ── Visualize search results ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))

# Plot all patches as grey
ax.scatter(
    result.patch_coords[:, 1], result.patch_coords[:, 0],
    c="lightgrey", s=10, marker="s", alpha=0.3,
)

# Highlight matches
match_indices = [m["index"] for m in matches]
match_distances = [m["distance"] for m in matches]
ax.scatter(
    result.patch_coords[match_indices, 1],
    result.patch_coords[match_indices, 0],
    c=match_distances, cmap="RdYlGn_r", s=80, marker="s",
    edgecolors="black", linewidths=1,
)

# Mark query
ax.scatter(
    [query_coords[1]], [query_coords[0]],
    c="red", s=150, marker="*", zorder=5, label="Query",
)

ax.set_title("Spatial Search Results")
ax.set_xlabel("X (pixels)")
ax.set_ylabel("Y (pixels)")
ax.invert_yaxis()
ax.set_aspect("equal")
ax.legend()
plt.tight_layout()
plt.show()

---
## 6. Cross-Dataset Comparison

Compare embeddings across multiple datasets for patient stratification. Load embeddings from multiple projects and visualize in a shared embedding space.

In [ ]:
from kintsugi.kronos.model import EmbeddingResult
from kintsugi.kronos.analysis import compare_datasets

# ── Load embeddings from multiple datasets ────────────────────────
# Uncomment and modify paths to your datasets
# dataset_dirs = [
#     Path("/blue/maigan/smith6jt/KINTSUGI_Projects/spleen_001"),
#     Path("/blue/maigan/smith6jt/KINTSUGI_Projects/lymph_node_001"),
#     Path("/blue/maigan/smith6jt/KINTSUGI_Projects/thymus_001"),
# ]
# dataset_names = ["Spleen", "Lymph Node", "Thymus"]
#
# results = []
# for d in dataset_dirs:
#     emb_path = d / "data" / "processed" / "embeddings" / "kronos_embeddings.h5"
#     if emb_path.exists():
#         results.append(EmbeddingResult.load(emb_path))
#     else:
#         print(f"No embeddings found for {d.name}, run extraction first")
#
# if len(results) >= 2:
#     fig = compare_datasets(
#         results,
#         dataset_names=dataset_names[:len(results)],
#         method="umap",
#         save_path=OUTPUT_DIR / "cross_dataset_comparison.png",
#     )
#     plt.show()

print("Uncomment the code above to compare multiple datasets.")
print("Each dataset needs embeddings extracted first (Section 3).")

---
## 7. Marker-Level Analysis

Analyze per-marker embeddings to understand which markers contribute most to tissue organization.

In [ ]:
# ── Marker embedding analysis ─────────────────────────────────────
# marker_embeddings shape: (n_patches, n_markers, embedding_dim)
print(f"Marker embeddings shape: {result.marker_embeddings.shape}")
print(f"Markers: {result.marker_names}")

# Compute per-marker variance across patches
# Higher variance = marker differentiates more between tissue regions
marker_variance = np.var(result.marker_embeddings, axis=0).mean(axis=1)

# Sort by variance
sorted_idx = np.argsort(marker_variance)[::-1]

print("\nMarker embedding variance (higher = more discriminative):")
for i, idx in enumerate(sorted_idx):
    print(f"  {i+1}. {result.marker_names[idx]}: {marker_variance[idx]:.4f}")

In [ ]:
# ── Visualize marker contributions ────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
names = [result.marker_names[i] for i in sorted_idx]
values = marker_variance[sorted_idx]
ax.barh(range(len(names)), values, color="steelblue")
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names, fontsize=8)
ax.set_xlabel("Embedding Variance")
ax.set_title("Marker Discriminative Power (KRONOS Embeddings)")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "marker_variance.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 8. Export for Downstream Analysis

Export embeddings and cluster labels for use in other tools (scanpy, scimap, etc.).

In [ ]:
import pandas as pd

# ── Create a summary table ────────────────────────────────────────
df = pd.DataFrame({
    "patch_idx": range(result.n_patches),
    "row": result.patch_coords[:, 0],
    "col": result.patch_coords[:, 1],
    "cluster_leiden": labels,
    "cluster_kmeans": labels_km,
})

# Add UMAP coordinates
umap_coords = reduce_dimensions(result, method="umap")
df["umap_1"] = umap_coords[:, 0]
df["umap_2"] = umap_coords[:, 1]

# Save
csv_path = OUTPUT_DIR / "kronos_patch_summary.csv"
df.to_csv(csv_path, index=False)
print(f"Saved patch summary to {csv_path}")
print(f"\n{df.head(10)}")

In [ ]:
# ── Export as AnnData for scanpy integration ──────────────────────
import anndata as ad

adata = ad.AnnData(
    X=result.patch_embeddings,
    obs=df,
)
adata.obsm["X_umap"] = umap_coords
adata.obsm["spatial"] = result.patch_coords[:, :2].astype(float)  # row, col

h5ad_path = OUTPUT_DIR / "kronos_embeddings.h5ad"
adata.write(h5ad_path)
print(f"Saved AnnData to {h5ad_path}")
print(f"Shape: {adata.shape}")

---
## Summary

This notebook demonstrated:

1. **Marker mapping** — Matching KINTSUGI channel names to KRONOS's 175 known markers
2. **Embedding extraction** — Tiling registered images and running KRONOS inference
3. **Clustering** — Leiden and KMeans clustering of patch embeddings
4. **Visualization** — UMAP embedding space and spatial cluster maps
5. **Spatial search** — Finding similar tissue regions by embedding distance
6. **Cross-dataset comparison** — Patient stratification across tissue cohorts
7. **Marker analysis** — Identifying discriminative markers from embeddings
8. **Export** — Saving results as CSV, HDF5, and AnnData for downstream tools

### Next Steps

- **Label-efficient phenotyping**: Annotate 5-10 patches per cell type, train a linear classifier on KRONOS embeddings
- **Tissue microenvironment discovery**: Analyze cluster compositions to identify immune niches, tumor margins, etc.
- **Batch processing**: Run embedding extraction as a Snakemake rule across all 47 datasets
- **Artifact detection**: Use KRONOS embeddings to identify tissue folds, staining failures, and out-of-focus regions